# الفصل 62: المعماريات التالية لما بعد Transformer

> طبقة الحدس رأينا في الفصل 16 أن Mamba وSSM تطرح بديلاً جذريًا لـ Attention.

- **الجزء:** 12 — الاتجاهات المستقبلية
- **الفصل:** 62
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 61: المحاذاة المتقدمة وسلامة الأنظمة الفائقة
- **الوسوم:** Post-Transformer، RWKV، RetNet، World Models

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-12-future%2Fch62-post-transformer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-12-future%2Fch62-post-transformer.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch62-post-transformer.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس رأينا في الفصل 16 أن Mamba وSSM تطرح بديلاً جذريًا لـ Attention.

> **بيانات الفصل**
>
> - **الجزء:** 12 — الاتجاهات المستقبلية
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 61: المحاذاة المتقدمة وسلامة الأنظمة الفائقة


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Mixture-of-Depths (MoD): الحوسبة الديناميكية عبر العمق
- فهم نماذج على مستوى البايت (Byte-Level Models)
- فهم الهجائن الخطية: الجيل التالي من دمج SSM وAttention


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**
> رأينا في الفصل 16 أن Mamba وSSM تطرح بديلاً جذريًا لـ Attention. لكن 2024-2025 شهدا مزيدًا من الابتكارات الموجَّهة نحو هدف واحد: **تقليل "الإهدار الحسابي"** — فليس كل رمز يحتاج نفس كمية الحوسبة، وليست كل مهمة تحتاج Tokenization على مستوى الكلمات الفرعية. فهم هذه الاتجاهات يُجهِّزك لقراءة الورقات البحثية القادمة بسهولة بدل المفاجأة بكل معمارية جديدة.


## 2. الخريطة الذهنية


```text
ما بعد Transformer
├── كفاءة الحوسبة
│   ├── Mixture-of-Depths (MoD)
│   ├── Mixture-of-Experts (MoE)
│   └── التوجيه الديناميكي
├── تمثيل النص
│   ├── Byte-Level (BLT)
│   ├── Tokenizer-free
│   └── الرقع الديناميكية
├── كفاءة التسلسل
│   ├── Mamba/SSM
│   ├── Hybrid SSM-Attention (Jamba/Zamba)
│   └── Linear Attention
└── أسئلة القرار
    ├── هل تحتاج سياقًا طويلًا؟
    ├── هل العتاد يدعم kernels؟
    └── هل المكسب يبرر التعقيد؟
```


## 3. الشرح المتدرج

### 3.1 Mixture-of-Depths (MoD): الحوسبة الديناميكية عبر العمق

> 🟡 **طبقة الآلية**

فكرة MoE (الفصل 15) طبّقت التخصيص الانتقائي على **العرض** (أي خبير FFN يُفعَّل)؛ Mixture-of-Depths تطبّق نفس المبدأ على **العمق**: ليس كل رمز يحتاج المرور عبر كل طبقة من طبقات الشبكة.


In [ ]:
import torch
import torch.nn as nn

class MixtureOfDepthsBlock(nn.Module):
    """
    تطبيق مبسّط لفكرة Mixture-of-Depths (DeepMind, 2024):
    كل طبقة تُقرر - عبر Router - أي الرموز "تستحق" المرور عبر الحساب
    الكامل (Attention + FFN) وأيها يمكنه "تخطي" الطبقة (Residual مباشر).
    النتيجة: توفير حسابي كبير مع الحفاظ على الأداء، لأن بعض الرموز
    (كعلامات الترقيم أو الكلمات المتكررة السهلة) لا تحتاج معالجة عميقة.
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 capacity_fraction: float = 0.5):
        super().__init__()
        # capacity_fraction: نسبة الرموز التي "تمر" عبر الحساب الكامل
        # (مثال: 0.5 يعني فقط نصف الرموز في كل تسلسل تُعالَج بالكامل)
        self.capacity_fraction = capacity_fraction
        self.router = nn.Linear(d_model, 1)  # يُنتج درجة أهمية لكل رمز

        self.attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, d_model = x.shape
        capacity = int(seq_len * self.capacity_fraction)

        # 1. الـ Router يحسب درجة أهمية لكل رمز
        router_scores = self.router(x).squeeze(-1)  # [batch, seq_len]

        # 2. اختيار أعلى `capacity` رمز من حيث الأهمية لكل عيّنة في الدفعة
        top_k_scores, top_k_indices = torch.topk(
            router_scores, k=capacity, dim=-1
        )
        # ترتيب الفهارس للحفاظ على الترتيب الأصلي للتسلسل (مهم للـ causal attention)
        top_k_indices, _ = torch.sort(top_k_indices, dim=-1)

        output = x.clone()

        for b in range(batch_size):
            selected_indices = top_k_indices[b]
            selected_tokens = x[b, selected_indices].unsqueeze(0)  # [1, capacity, d]

            # 3. الرموز المختارة فقط تمر عبر Attention + FFN الكاملين
            attn_out, _ = self.attention(
                selected_tokens, selected_tokens, selected_tokens,
                attn_mask=self._causal_mask(capacity, x.device)
            )
            selected_tokens = self.norm1(selected_tokens + attn_out)
            ffn_out = self.ffn(selected_tokens)
            processed_tokens = self.norm2(selected_tokens + ffn_out)

            # 4. دمج الرموز المُعالَجة مرة أخرى في مواضعها الأصلية
            #    (الرموز غير المختارة تمرّ عبر Residual فقط - بدون معالجة)
            output[b, selected_indices] = processed_tokens.squeeze(0)

        return output

    def _causal_mask(self, size, device):
        return torch.triu(
            torch.full((size, size), float('-inf'), device=device), diagonal=1
        )


> ⚖️ **MoE مقابل MoD**

| المعيار | Mixture-of-Experts (MoE) | Mixture-of-Depths (MoD) |
|---|---|---|
| محور التخصيص | أي FFN "خبير" يُستخدم (العرض) | هل الرمز يمر عبر الطبقة أصلًا (العمق) |
| التوفير | في عدد المعاملات الفعّالة لكل رمز | في عدد الطبقات الفعّالة لكل رمز |
| التعقيد التنفيذي | معتدل (موجود في أطر العمل الكبرى) | أعلى (توزيع ديناميكي يصعب موازنته على GPU بكفاءة) |
| إمكانية الدمج | — | **تُدمَج مع MoE** لنتائج تراكمية (MoDE) |

### 3.2 نماذج على مستوى البايت (Byte-Level Models)

> 🟡 **طبقة الآلية**

كل النماذج التي درسناها تعتمد على Tokenizer (BPE/WordPiece - الفصل 6) يُحوِّل النص لرموز فرعية. لكن هذا يخلق مشاكل: تحيّز لغوي (كلمات العربية والهندية وغيرها من اللغات غير الإنجليزية غالبًا "تتكسّر" لرموز أكثر، مما يرفع التكلفة ويُقلّص السياق الفعّال)، ومشاكل مع الأخطاء الإملائية والنصوص غير المعيارية.

**BLT (Byte Latent Transformer - Meta, 2024)** يطرح حلاً: العمل مباشرة على البايتات الخام، لكن بذكاء حسابي يتجنب الانفجار في طول التسلسل.


In [ ]:
class DynamicBytePatching:
    """
    تطبيق مفاهيمي لفكرة BLT (Byte Latent Transformer):
    بدلاً من Tokenizer ثابت المفردات، تُجمَّع البايتات في "رقع" (Patches)
    ديناميكية الطول بناءً على التعقيد المحلي للنص (Entropy-based)،
    مقاسةً بواسطة نموذج لغوي صغير يعمل على مستوى البايت.
    """
    def __init__(self, byte_lm, entropy_threshold: float = 1.5):
        self.byte_lm = byte_lm  # نموذج صغير يتنبأ بالبايت التالي
        self.entropy_threshold = entropy_threshold

    def compute_byte_entropy(self, byte_sequence: bytes) -> list:
        """حساب إنتروبيا التنبؤ بكل بايت بناءً على ما سبقه (عدم اليقين)."""
        entropies = []
        for i in range(1, len(byte_sequence)):
            context = byte_sequence[:i]
            next_byte_probs = self.byte_lm.predict_next_byte_distribution(context)
            entropy = -sum(p * torch.log(p + 1e-10) for p in next_byte_probs)
            entropies.append(entropy.item())
        return entropies

    def create_dynamic_patches(self, byte_sequence: bytes) -> list:
        """
        إنشاء رقع ديناميكية: حد رقعة جديدة يُفتَح عند نقاط عدم اليقين
        العالي (إنتروبيا مرتفعة = بداية كلمة/مفهوم جديد محتمل)،
        بينما تُجمَّع البايتات شديدة القابلية للتنبؤ (مثل نهايات الكلمات
        الشائعة) في رقعة واحدة أطول. هذا يُنتج ضغطًا تكيفيًا: نص بسيط
        متكرر → رقع طويلة (حوسبة أقل)، نص معقد/نادر → رقع قصيرة (حوسبة أدق).
        """
        entropies = self.compute_byte_entropy(byte_sequence)
        patches = []
        current_patch_start = 0

        for i, entropy in enumerate(entropies):
            if entropy > self.entropy_threshold:
                patches.append(byte_sequence[current_patch_start:i+1])
                current_patch_start = i + 1

        if current_patch_start < len(byte_sequence):
            patches.append(byte_sequence[current_patch_start:])

        return patches


> 💡 **نصيحة عملية**: لو كان تطبيقك يتعامل بكثافة مع نصوص عربية (أو أي لغة غير إنجليزية) بالإضافة لأكواد برمجية ونصوص مختلطة (كود + تعليقات عربية)، راقب "fertility" (الفصل 19) لـ Tokenizer الحالي عن كثب — نماذج مستوى-البايت الناشئة قد تحل مشكلة التحيّز اللغوي جذريًا خلال السنوات القادمة لكنها ليست إنتاجية الجاهزية بعد على نطاق واسع (2025).

### 3.3 الهجائن الخطية: الجيل التالي من دمج SSM وAttention

> 🔴 **طبقة التعمّق**

منذ الفصل 16 (Mamba/Jamba)، تطوّرت الهجائن بشكل ملحوظ. **Zamba وHymba وJamba 1.5** تُحسِّن استراتيجية الدمج:


In [ ]:
class AdaptiveHybridBlock(nn.Module):
    """
    معمارية هجينة متقدمة (بأسلوب Hymba/Zamba 2024): بدلاً من تبديل طبقات
    Attention وMamba بالتناوب كما في Jamba البسيط، تُدمَج مساراتهما
    بالتوازي (Parallel Heads) داخل نفس الطبقة، مع دمج مُرجَّح يسمح
    للنموذج بتعلّم متى يعتمد على كل مسار (ذاكرة دقيقة قصيرة المدى عبر
    Attention مقابل ملخّص مُضغوط طويل المدى عبر SSM).
    """
    def __init__(self, d_model: int, n_attn_heads: int, ssm_state_dim: int):
        super().__init__()
        # مسار Attention الدقيق (يحتفظ بتفاصيل كاملة - محدود بطول نافذة محلية)
        self.local_attention = nn.MultiheadAttention(
            d_model, n_attn_heads, batch_first=True
        )
        self.local_window_size = 256  # نافذة محلية فقط - اقتصاد حسابي

        # مسار SSM (يحتفظ بملخص مُضغوط للسياق الكامل بلا حدود عملية)
        from_chapter_16_mamba_block = None  # راجع SimplifiedMambaBlock (الفصل 16)
        self.ssm_path = SimplifiedMambaBlock(d_model, state_dim=ssm_state_dim)

        # بوابة دمج مُتعلَّمة (Learned Fusion Gate)
        self.fusion_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # معالجة متوازية (وليست متسلسلة) للمسارين
        local_out = self._windowed_attention(x)
        ssm_out = self.ssm_path(x)

        # بوابة تُقرر نسبة المزج بين المسارين لكل رمز ديناميكيًا
        gate_input = torch.cat([local_out, ssm_out], dim=-1)
        gate = self.fusion_gate(gate_input)
        fused = gate * local_out + (1 - gate) * ssm_out

        return self.norm(x + fused)

    def _windowed_attention(self, x):
        # تطبيق Attention ضمن نافذة محلية منزلقة فقط (اقتصاد O(n·w) بدل O(n²))
        # تفاصيل Sliding Window Attention غُطيت ضمنيًا في الفصل 14
        return x  # تبسيط توضيحي


### 3.4 جدول مقارنة شامل: اتجاهات ما بعد Transformer

| الاتجاه | المشكلة المستهدفة | التوفير الرئيسي | مدى النضج الإنتاجي (2025) |
|---|---|---|---|
| Mixture-of-Depths | إهدار حسابي عبر الطبقات | حتى 50% من FLOPs للتمرير الأمامي | بحثي، تجريبي |
| Byte-Level (BLT) | تحيّز Tokenizer اللغوي | يُلغي الحاجة لـ Tokenizer كليًا | بحثي مبكر جدًا |
| Hybrid SSM-Attention (Jamba/Zamba) | O(n²) من Attention البحت | ذاكرة KV-Cache أصغر بكثير | **إنتاجي فعليًا** (Jamba متاح تجاريًا) |
| MoE (مُراجَعة) | حجم معاملات ثابت لكل الرموز | حوسبة أقل لكل رمز رغم حجم إجمالي أكبر | **إنتاجي راسخ** (Mixtral, DeepSeek-V3, GPT-4) |
| نماذج الاستدلال (الفصل 60) | جودة الإجابة لمهام معقدة | — (يزيد الحوسبة عمدًا) | **إنتاجي فعليًا** (o1/o3/R1) |


## 4. مثال محلول خطوة بخطوة

**مثال 1: توفير MoD.**

نموذج كثيف: 32 طبقة، كل رمز يمر عبر كل الطبقات. إذا كان لدينا 1,000 رمز، فعدد «تمريرات الرمز-الطبقة» = \(32 \times 1{,}000 = 32{,}000\).

MoD بنسبة سعة 50%: كل طبقة تعالج 500 رمزًا فقط:

\[
32 \times 500 = 16{,}000
\]

توفير 50% في الحوسبة الأساسية، مع إضافة router صغير (نحو 2% overhead). التوفير الصافي ≈ 48%.

**لكن:** التنفيذ على GPU قد لا يحقق التوفير بسبب التوزيع الديناميكي وعدم انتظام الدفعات. القياس الفعلي ضروري.

**مثال 2: Byte-Level للعربية.**

| النص | BPE tokens (تقريبي) | Bytes |
|---|---:|---:|
| «المستشفى» | 3-4 | 12 |
| «استقلالية» | 4-5 | 14 |
| جملة 100 كلمة | 250 token | 700 byte |

التسلسل البايتي أطول 2-3 مرات، لكنه يلغي تحيّز التقطيع. BLT يعالج ذلك برقع ديناميكية تُجمّع البايتات المتوقعة، فيقلّل الطول الفعّال. المكسب الحقيقي في اللغات المهمّشة من tokenizers الإنجليزية، لكن التكلفة الحسابية ما زالت عائقًا.


## 5. نصائح وحيل

> 💡 **ابنِ خط أساس قويًا أولًا**: Transformer محسّن (RoPE+GQA+RMSNorm+SwiGLU) يظل الخيار الإنتاجي الآمن. قارن أي بديل به.

> 💡 **افحص توفر kernels**: معمارية أذكى نظريًا قد تكون أبطأ على GPU إن لم تكن لها نوى محسّنة. لا تعتمد على أرقام الورقة فقط.

> 💡 **قِس على مهامك**: MoD قد يوفّر FLOPs لكنه يزيد التعقيد. اختبر على مهمة حقيقية، لا على benchmark عام فقط.

> 💡 **راقب fertility للعربية**: إذا كان tokenizer يستهلك 2-3 أضعاف رموز الإنجليزية، فbyte-level أو tokenizer مخصص يستحق الدراسة.

> 💡 **لا تتبنَّ معمارية بحثية في الإنتاج دون خطة تراجع**: احتفظ بخط أساس قابل للتشغيل، واختبر البديل في ظل حملك.


## 6. أخطاء شائعة ومصائد

- افتراض أن كل معمارية "جديدة" في ورقة بحثية جاهزة للإنتاج فورًا — معظم الاتجاهات في هذا الفصل (باستثناء الهجائن وMoE) لا تزال في طور البحث النشط ولم تُثبت بعد على نطاق مليارات المعاملات بنفس موثوقية Transformer القياسي.
- تجاهل تكلفة **الهندسة الحاسوبية** (Hardware Utilization) عند تقييم معمارية جديدة؛ معمارية "أذكى نظريًا" قد تكون أبطأ عمليًا على GPU لأنها لا تستغل التوازي المصفوفي الكثيف الذي يُحسِّنه CUDA جيدًا (مشكلة شائعة جدًا مع MoD وتوزيعات الحوسبة الديناميكية).


## 7. جدول مقارنة

| المعمارية | تعقيد التسلسل | تعقيد الذاكرة | النضج الإنتاجي (2025) | الأنسب لـ |
|---|---|---|---|---|
| Transformer قياسي | \(O(n^2)\) | KV Cache كبير | راسخ | الاستخدام العام |
| MoE | \(O(n^2)\) مع FFN متناثر | كبير (كل الخبراء) | راسخ | نماذج ضخمة بكفاءة حسابية |
| Mamba/SSM | \(O(n)\) | ثابت | متوسط | سياقات طويلة، تدفق |
| Jamba/Zamba (هجين) | \(O(n)\) مع Attention محلي | أصغر | إنتاجي جزئيًا | سياق طويل + جودة Attention |
| BLT (Byte-Level) | يعتمد على الرقع | متوسط | بحثي مبكر | لغات مهمّشة، نصوص غير معيارية |
| MoD | \(O(n^2)\) مع عمق ديناميكي | متوسط | بحثي | تقليل FLOPs |


## 8. ملخص أفضل الممارسات

- ✅ لبناء نموذج إنتاجي اليوم (2025): اعتمد على Transformer القياسي المُحسَّن (RoPE+GQA+RMSNorm+SwiGLU - الفصل 17) كخط أساس آمن، وفكّر في Hybrid SSM فقط إذا كان طول السياق المستهدف ضخمًا جدًا (>128K) وكانت تكلفة KV-Cache عائقًا حقيقيًا.
- ✅ تابع benchmark مثل "Long Context Arena" و"RULER" عند تقييم ادعاءات السياق الطويل لأي معمارية جديدة.
- ✅ لا تتبنَّ معمارية بحثية جديدة في الإنتاج دون التحقق من وجود تطبيق مُحسَّن لـ GPU (CUDA kernels) لها — وإلا ستخسر كل المكاسب النظرية بسبب بطء التنفيذ الفعلي.


## 9. أسئلة مقابلات

**سؤال:** لماذا تُعتبر نماذج مستوى-البايت جذابة نظريًا للغة العربية تحديدًا، وما أكبر عائق عملي أمام تبنّيها حاليًا؟

> **إجابة نموذجية:** العربية (وبشكل عام اللغات ذات الأنظمة الكتابية غير اللاتينية والصرف الغني) تعاني من fertility مرتفع في Tokenizers المدرَّبة أساسًا على نصوص إنجليزية مهيمنة — كلمة عربية واحدة قد "تتكسّر" إلى عدة رموز فرعية، مما يُقلِّص طول السياق الفعّال المتاح ويرفع تكلفة الاستدلال نسبيًا لكل وحدة معنى. النماذج على مستوى البايت تُلغي هذا التحيّز كليًا لأنها لا تعتمد على مفردات مُدرَّبة مسبقًا بل على تمثيل عالمي (256 قيمة بايت ممكنة لأي لغة). العائق الأكبر: طول التسلسل الخام للبايتات أطول بكثير من التسلسل بعد BPE، مما يتطلب آليات ضغط ديناميكي ذكية (كالـ Entropy-based Patching في BLT) لتفادي انفجار التكلفة الحسابية التربيعية لـ Attention؛ هذه الآليات لا تزال حديثة وغير مُحسَّنة بنفس مستوى نضج BPE/SentencePiece الراسخين.


## 10. تمرين عملي

**الهدف:** محاكاة MoD على مهمة نسخ بسيطة.

**الخطوات:**
1. نفّذ Transformer صغيرًا (4 طبقات) لمهمة copy.
2. أضف router يختار 50% من الرموز لكل طبقة.
3. درّب النسختين على نفس البيانات والميزانية.
4. قِس: دقة النسخ، عدد FLOPs التقريبي، وزمن التدريب.
5. أعد التجربة بنسب سعة 25% و75%.
6. اكتب استنتاجًا: هل التوفير يبرر فقدان الدقة؟

**معيار النجاح:** جدول يوضح المقايضة بين السعة والدقة والحوسبة، مع تفسير.


## 11. مراجع للتعمّق

- Raposo et al. (2024, DeepMind): *Mixture-of-Depths: Dynamically allocating compute in transformer-based language models*
- Pagnoni et al. (2024, Meta): *Byte Latent Transformer: Patches Scale Better Than Tokens*
- Glorioso et al. (2024): *Zamba: A Compact 7B SSM Hybrid Model*
- Dong et al. (2024): *Hymba: A Hybrid-head Architecture for Small Language Models*
- Lieber et al. (2024, AI21): *Jamba-1.5: Hybrid Transformer-Mamba Models at Scale*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — الاتجاهات المستقبلية / الفصل 62: المعماريات التالية لما بعد Transformer.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
